# KNN / WKNN trên UJIIndoorLoc

Chọn một hoặc nhiều model KNN/WKNN; notebook in thông số và vẽ bảng, CDF, sai số theo tòa, thời gian và ma trận nhầm lẫn **tầng**. Repo private: dùng Colab Secret `GITHUB_TOKEN` có quyền **Contents: Read**, hoặc upload ZIP của nhánh. Không ghi token vào notebook hay output. Để báo cáo kết quả chính thức, chọn full; smoke chỉ kiểm tra luồng.


In [ ]:
# 1. Tải repo private. Sau khi có commit mới, chạy lại ô này.
import io
import shutil
import subprocess
import urllib.error
import urllib.request
import zipfile
from pathlib import Path
from google.colab import files

REPO = "VietUniverse/ips-wifi-research"
BRANCH = "feature/knn-wknn"  # sau khi PR được merge có thể đổi thành main
ZIP_PATH = None  # hoặc đường dẫn ZIP đã upload, ví dụ "/content/ips-wifi-research.zip"

repo_zip = Path(ZIP_PATH).read_bytes() if ZIP_PATH else None
if repo_zip is None:
    try:
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        token = None
    if token:
        try:
            request = urllib.request.Request(
                f"https://api.github.com/repos/{REPO}/zipball/{BRANCH}",
                headers={"Authorization": f"Bearer {str(token).strip()}",
                         "Accept": "application/vnd.github+json"})
            with urllib.request.urlopen(request, timeout=120) as response:
                repo_zip = response.read()
        except urllib.error.HTTPError as error:
            print(f"GitHub trả HTTP {error.code}. Kiểm tra token có quyền Contents: Read,"
                  " đã chọn đúng repo, bật quyền truy cập Secret cho notebook và BRANCH tồn tại.")
        except urllib.error.URLError:
            print("Không kết nối được GitHub API. Có thể dùng ZIP của repo để tiếp tục.")
    token = None
if repo_zip is None:
    print("Upload ZIP của đúng nhánh repo từ GitHub (Code → Download ZIP).")
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Hãy upload đúng một file ZIP")
    repo_zip = next(iter(uploaded.values()))

root = Path("/content/knn_repo_source")
if root.exists():
    shutil.rmtree(root)
root.mkdir()
with zipfile.ZipFile(io.BytesIO(repo_zip)) as archive:
    archive.extractall(root)
projects = list(root.rglob("pyproject.toml"))
if len(projects) != 1:
    raise RuntimeError(f"ZIP cần đúng một pyproject.toml; tìm thấy {len(projects)}")
project = projects[0].parent
if not (project / "src/ips/neighbors").is_dir():
    raise RuntimeError("Nhánh/ZIP này chưa có KNN. Chọn BRANCH=feature/knn-wknn hoặc ZIP của nhánh KNN.")
print("Project:", project)
subprocess.run(["python", "-m", "pip", "install", "-e", str(project)], check=True)
subprocess.run(["python", "-m", "ips.experiments.knn_cli", "--list-options"], check=True)


In [ ]:
# 2. Tải UJIIndoorLoc từ UCI nếu chưa có hai CSV.
import urllib.request

data_dir = Path("/content/data/uji")
data_dir.mkdir(parents=True, exist_ok=True)
if not all((data_dir / name).exists() for name in ("trainingData.csv", "validationData.csv")):
    zip_path = Path("/content/ujiindoorloc.zip")
    urllib.request.urlretrieve("https://archive.ics.uci.edu/static/public/310/ujiindoorloc.zip", zip_path)
    extracted = Path("/content/uji_source")
    extracted.mkdir(exist_ok=True)
    with zipfile.ZipFile(zip_path) as archive:
        archive.extractall(extracted)
    for name in ("trainingData.csv", "validationData.csv"):
        matches = list(extracted.rglob(name))
        if len(matches) != 1:
            raise RuntimeError(f"Không tìm thấy duy nhất {name} trong ZIP UJI")
        shutil.copy2(matches[0], data_dir / name)
print("UJI data:", [(p.name, p.stat().st_size) for p in data_dir.glob("*.csv")])


In [ ]:
# 3. Chọn một hoặc nhiều model; smoke kiểm tra luồng, full dùng toàn bộ dữ liệu.
from datetime import datetime, timezone
import json
from ips.neighbors import available_models
from IPython.display import display

print('Model khả dụng:', sorted(available_models()))
MODELS = sorted(available_models())  # Ví dụ chỉ chạy một: ['knn']
CONFIG = "configs/knn_smoke.json"  # đổi thành configs/knn_full.json khi sẵn sàng
settings = json.loads((project / CONFIG).read_text(encoding='utf-8'))
if not MODELS or any(m not in available_models() for m in MODELS):
    raise ValueError('MODELS phải là các model trong danh sách khả dụng')
settings['models'] = MODELS
selected_config = Path('/content/selected_knn_config.json')
selected_config.write_text(json.dumps(settings, indent=2), encoding='utf-8')
print('Cấu hình:', json.dumps(settings, ensure_ascii=False, indent=2))
print('Số dòng dữ liệu:', [(p.name, sum(1 for _ in p.open()) - 1) for p in data_dir.glob('*.csv')])
run_dir = Path("/content/runs") / datetime.now(timezone.utc).strftime("knn_%Y%m%d_%H%M%S_%f")
subprocess.run(["python", "-m", "ips.experiments.knn_cli", "--config", str(selected_config),
                "--data-dir", str(data_dir), "--output-dir", str(run_dir)], check=True)
print((run_dir / "summary.json").read_text(encoding="utf-8"))
from ips.experiments.visualize import render_run
report_table, report_figures = render_run(run_dir)
print('Ảnh báo cáo:', [str(p) for p in report_figures])


In [ ]:
# 4. Tải kết quả về máy.
archive_path = shutil.make_archive(str(run_dir), "zip", root_dir=run_dir)
print("Result ZIP:", archive_path)
files.download(archive_path)
